In [7]:
import numpy as np

```python
self.w_1 = np.random.randn(hidden_size, n_features) * 0.1
```

`w_1` is computed here as a matrix of `hidden_size` rows and `n_features` columns. These represent the weight we need for each feature per layer. `0.1` multiplication just to scale

In [4]:

# --- Activation functions ---
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_cross_entropy(y_true, y_pred):
    epsilon = 1e-12  # to avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


```python
self.w_2 -= self.lr * (delta_2[:, None] @ self.a_1[None])
```
This is applying the derivative of the ReLU activation function. So `z1 > 0` returns a boolean array (same shape as z1) where:

- True (1) where z1 > 0
- False (0) where z1 <= 0

Multiplying by this boolean mask is equivalent to:

- Passing the gradient backward only through the neurons that were "active" (i.e., where z1 > 0)
- Blocking gradient flow where the neuron was inactive (i.e., z1 <= 0)

This is essential for correct backpropagation when using ReLU. If you skip this step:

- You'll update neurons that were "off" (output was 0), which is wrong.
- It can cause training to diverge or make learning inefficient.

In [5]:
class BinaryClassifier:
    def __init__(self, hidden_size: int, n_features: int):
        self.w_1 = np.random.randn(n_features, hidden_size) * 0.1
        self.b_1 = np.zeros((1, hidden_size))

        self.w_2 = np.random.randn(hidden_size, 1) * 0.1
        self.b_2 = np.zeros((1,))

    def forward(self, X):
        z_1 = X @ self.w_1 + self.b_1       # shape: (batch_size, hidden)
        a_1 = relu(z_1)                     # shape: (batch_size, hidden)
        z_2 = a_1 @ self.w_2 + self.b_2     # shape: (batch_size, 1)
        a_2 = sigmoid(z_2)                  # shape: (batch_size, 1)
        return z_1, a_1, z_2, a_2

    def backward(self, X, y, z_1, a_1, z_2, a_2, lr):
        m = X.shape[0]  # number of samples

        delta_2 = a_2 - y.reshape(-1, 1)              # shape: (m, 1)
        d_w2 = a_1.T @ delta_2 / m                    # shape: (hidden, 1)
        d_b2 = np.mean(delta_2, axis=0)               # shape: (1,)

        delta_1 = (delta_2 @ self.w_2.T) * relu_derivative(z_1)
        d_w1 = X.T @ delta_1 / m                      # shape: (features, hidden)
        d_b1 = np.mean(delta_1, axis=0, keepdims=True)

        self.w_2 -= lr * d_w2
        self.b_2 -= lr * d_b2
        self.w_1 -= lr * d_w1
        self.b_1 -= lr * d_b1

    def train(self, X, y, epochs=100, lr=0.01):
        for epoch in range(epochs):
            z_1, a_1, z_2, a_2 = self.forward(X)
            loss = binary_cross_entropy(y.reshape(-1, 1), a_2)

            preds = (a_2 >= 0.5).astype(int).flatten()
            acc = np.mean(preds == y)

            print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Accuracy: {acc:.2f}")
            self.backward(X, y, z_1, a_1, z_2, a_2, lr)

    def predict(self, X):
        _, _, _, a_2 = self.forward(X)
        return (a_2 >= 0.5).astype(int).flatten()


In [ ]:
# Create fake dataset: 100 samples, 5 features
np.random.seed(0)
X_data = np.random.randn(100, 5)
true_weights = np.array([1.5, -2, 0.7, 0.1, -1.2])
y_data = (X_data @ true_weights + np.random.randn(100) * 0.5 > 0).astype(int)

# Model
model = BinaryClassifier(hidden_size=8, n_features=5)
model.train(X_data, y_data, epochs=100, lr=0.1)

# Predictions
preds = model.predict(X_data)
print(f"Final Accuracy: {np.mean(preds == y_data):.2f}")


In [ ]:
from mnist.helper import split_by_label_binary

x_train, y_train, x_test, y_test = split_by_label_binary(0, "./data/")

In [19]:
model = BinaryClassifier(hidden_size=8, n_features=784)
model.train(x_train, y_train, epochs=150, lr=0.1)

Epoch 1 | Loss: 0.6936 | Accuracy: 0.48
Epoch 2 | Loss: 0.6853 | Accuracy: 0.51
Epoch 3 | Loss: 0.6771 | Accuracy: 0.54
Epoch 4 | Loss: 0.6686 | Accuracy: 0.59
Epoch 5 | Loss: 0.6597 | Accuracy: 0.62
Epoch 6 | Loss: 0.6501 | Accuracy: 0.65
Epoch 7 | Loss: 0.6396 | Accuracy: 0.67
Epoch 8 | Loss: 0.6283 | Accuracy: 0.69
Epoch 9 | Loss: 0.6161 | Accuracy: 0.70
Epoch 10 | Loss: 0.6031 | Accuracy: 0.72
Epoch 11 | Loss: 0.5894 | Accuracy: 0.74
Epoch 12 | Loss: 0.5750 | Accuracy: 0.75
Epoch 13 | Loss: 0.5601 | Accuracy: 0.77
Epoch 14 | Loss: 0.5446 | Accuracy: 0.79
Epoch 15 | Loss: 0.5287 | Accuracy: 0.80
Epoch 16 | Loss: 0.5124 | Accuracy: 0.82
Epoch 17 | Loss: 0.4958 | Accuracy: 0.84
Epoch 18 | Loss: 0.4789 | Accuracy: 0.85
Epoch 19 | Loss: 0.4619 | Accuracy: 0.86
Epoch 20 | Loss: 0.4449 | Accuracy: 0.87
Epoch 21 | Loss: 0.4280 | Accuracy: 0.88
Epoch 22 | Loss: 0.4114 | Accuracy: 0.89
Epoch 23 | Loss: 0.3951 | Accuracy: 0.90
Epoch 24 | Loss: 0.3793 | Accuracy: 0.90
Epoch 25 | Loss: 0.3642 |

In [20]:
preds = model.predict(x_test)
print(f"Final Accuracy: {np.mean(preds == y_test):.2f}")

Final Accuracy: 0.97
